---
## ⚠️ DEPRECATION NOTICE

**This notebook is the original version and has been superseded by a refactored version.**

Please consider using the refactored version instead:
- **`ML - Sleep State Detection - Refactored.ipynb`** - Uses the new modular `src/` package

### Benefits of the Refactored Version:
- ✓ Modular, reusable code organized in the `src/` package
- ✓ Comprehensive documentation and type hints
- ✓ Centralized configuration management
- ✓ Better error handling and validation
- ✓ Easier to maintain and test

This original notebook is kept for reference purposes.

---


## 1. Data Import

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report 

path1 = "/kaggle/input/child-mind-institute-detect-sleep-states/train_series.parquet"
train_series = pd.read_parquet(path1)
print(train_series.head(100))


path2 = "/kaggle/input/child-mind-institute-detect-sleep-states/train_events.csv"
train_events = pd.read_csv(path2)

print(train_events.head(100))

descriptive_stats = train_series.describe()
print(descriptive_stats)


/opt/conda/lib/python3.10/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.23.5
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


       series_id  step                 timestamp     anglez    enmo
0   038441c925bb     0  2018-08-14T15:30:00-0400   2.636700  0.0217
1   038441c925bb     1  2018-08-14T15:30:05-0400   2.636800  0.0215
2   038441c925bb     2  2018-08-14T15:30:10-0400   2.637000  0.0216
3   038441c925bb     3  2018-08-14T15:30:15-0400   2.636800  0.0213
4   038441c925bb     4  2018-08-14T15:30:20-0400   2.636800  0.0215
..           ...   ...                       ...        ...     ...
95  038441c925bb    95  2018-08-14T15:37:55-0400 -80.013603  0.0128
96  038441c925bb    96  2018-08-14T15:38:00-0400 -80.007599  0.0136
97  038441c925bb    97  2018-08-14T15:38:05-0400 -80.136703  0.0135
98  038441c925bb    98  2018-08-14T15:38:10-0400 -80.111801  0.0132
99  038441c925bb    99  2018-08-14T15:38:15-0400 -80.051201  0.0130

[100 rows x 5 columns]
       series_id  night   event     step                 timestamp
0   038441c925bb      1   onset   4992.0  2018-08-14T22:26:00-0400
1   038441c925bb      1  w

## 2. Data Cleansing

In [2]:
# Get the total number of rows in train_series
total_rows_train_series = train_series.shape[0]
print("Total number of rows in train_series:", total_rows_train_series)

# Get the total number of rows in train_events
total_rows_train_events = train_events.shape[0]
print("Total number of rows in train_events:", total_rows_train_events)


# Check for NaN values in train_series
nan_rows_train_series = train_series[train_series.isna().any(axis=1)]
print("Number of rows with NaN values in train_series:", len(nan_rows_train_series))

# Check for NaN values in train_events
nan_rows_train_events = train_events[train_events.isna().any(axis=1)]
print("Number of rows with NaN values in train_events:", len(nan_rows_train_events))

Total number of rows in train_series: 127946340
Total number of rows in train_events: 14508
Number of rows with NaN values in train_series: 0
Number of rows with NaN values in train_events: 4923


In [3]:
# Drop rows with any NaN values in train_events
train_events_cleaned = train_events.dropna()

# Get the total number of rows in train_series after cleansing
total_rows_train_events = train_events_cleaned.shape[0]
print("Total number of rows in train_events:", total_rows_train_events)

# Check for NaN values in train_series after cleansing
nan_rows_train_events = train_events_cleaned[train_series.isna().any(axis=1)]
print("Number of rows with NaN values in train_events:", len(nan_rows_train_events))

Total number of rows in train_events: 9585
Number of rows with NaN values in train_events: 0


/tmp/ipykernel_26/2473540693.py:9: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  nan_rows_train_events = train_events_cleaned[train_series.isna().any(axis=1)]


## 3. Merging train_series and train_events

In [4]:
# Merge the datasets based on series_id and step
merged_data = pd.merge(train_series, train_events, on=['series_id', 'step', 'timestamp'])
print(merged_data.head(100))

total_rows_merged_data = merged_data.shape[0]
print("Total number of rows in merged_data:", total_rows_merged_data)


       series_id    step                 timestamp     anglez    enmo  night  \
0   038441c925bb    4992  2018-08-14T22:26:00-0400 -78.690598  0.0099      1   
1   038441c925bb   10932  2018-08-15T06:41:00-0400 -61.578201  0.0263      1   
2   038441c925bb   20244  2018-08-15T19:37:00-0400  -6.387400  0.0182      2   
3   038441c925bb   27492  2018-08-16T05:41:00-0400 -45.355099  0.0165      2   
4   038441c925bb   39996  2018-08-16T23:03:00-0400  -1.786700  0.0000      3   
..           ...     ...                       ...        ...     ...    ...   
95  04f547b8017d   82404  2018-12-03T06:27:00-0500  49.896000  0.0000      5   
96  04f547b8017d   92892  2018-12-03T21:01:00-0500  27.531099  0.0062      6   
97  04f547b8017d   99672  2018-12-04T06:26:00-0500  -6.061800  0.0259      6   
98  04f547b8017d  110244  2018-12-04T21:07:00-0500  10.982900  0.0000      7   
99  04f547b8017d  117540  2018-12-05T07:15:00-0500 -52.872601  0.0000      7   

     event  
0    onset  
1   wakeup  


In [5]:
# clear memory

del train_series
import gc
gc.collect()

0

## 4. Feature Engineering

### 4.1 Time Information

In [6]:
# Remove timezone information from the timestamp (e.g., "-0400")
merged_data['timestamp_notimezone'] = merged_data['timestamp'].str[:-5]

# Convert timestamp to datetime
merged_data['timestamp_utc'] = pd.to_datetime(merged_data['timestamp_notimezone'],utc=True)

# Extract time from timestamp
merged_data['time'] = merged_data['timestamp_utc'].dt.time

# Extract time from timestamp
merged_data['hour'] = merged_data['timestamp_utc'].dt.hour

# Extract day of the week from timestamp and convert to string (e.g., 'Monday', 'Tuesday', etc.)
merged_data['day_of_week'] = merged_data['timestamp_utc'].dt.day_name()

print(merged_data.head(100))


       series_id    step                 timestamp     anglez    enmo  night  \
0   038441c925bb    4992  2018-08-14T22:26:00-0400 -78.690598  0.0099      1   
1   038441c925bb   10932  2018-08-15T06:41:00-0400 -61.578201  0.0263      1   
2   038441c925bb   20244  2018-08-15T19:37:00-0400  -6.387400  0.0182      2   
3   038441c925bb   27492  2018-08-16T05:41:00-0400 -45.355099  0.0165      2   
4   038441c925bb   39996  2018-08-16T23:03:00-0400  -1.786700  0.0000      3   
..           ...     ...                       ...        ...     ...    ...   
95  04f547b8017d   82404  2018-12-03T06:27:00-0500  49.896000  0.0000      5   
96  04f547b8017d   92892  2018-12-03T21:01:00-0500  27.531099  0.0062      6   
97  04f547b8017d   99672  2018-12-04T06:26:00-0500  -6.061800  0.0259      6   
98  04f547b8017d  110244  2018-12-04T21:07:00-0500  10.982900  0.0000      7   
99  04f547b8017d  117540  2018-12-05T07:15:00-0500 -52.872601  0.0000      7   

     event timestamp_notimezone        

### 4.2 Sliding Window for anglez and enmo

In [7]:
# Sort the DataFrame based on timestamp
merged_data = merged_data.sort_values(by='time')

print(merged_data.head(100))

         series_id    step                 timestamp     anglez    enmo  \
5203  8e32047cbc1f   39600  2017-09-18T00:00:00-0400 -70.946701  0.0223   
1680  29c75c018220  198360  2018-04-16T00:00:00-0400 -27.437000  0.0105   
1694  29c75c018220  319320  2018-04-23T00:00:00-0400 -37.416100  0.0173   
2967  4feda0596965  231492  2018-05-26T00:01:00-0400  16.277399  0.0025   
7401  ce9164297046  230772  2018-02-07T00:01:00-0500  12.339900  0.0042   
...            ...     ...                       ...        ...     ...   
3627  5ffd5e1e81ac  561528  2019-01-08T00:09:00-0500  43.518299  0.0000   
3471  5f40907ec171  298008  2017-08-29T00:09:00-0400  -5.007400  0.0252   
4473  76237b9406d5  143928  2017-11-23T00:09:00-0500 -34.427799  0.0077   
3917  694faf956ebf  164808  2019-01-27T00:09:00-0500  72.652603  0.0097   
2106  3452b878e596   94788  2019-06-02T00:09:00-0400 -60.842499  0.0000   

      night  event timestamp_notimezone             timestamp_utc      time  \
5203      3  onset  

In [8]:
window_size = 10

# Calculate the rolling average of anglez
merged_data['anglez_rolling_avg'] = merged_data['anglez'].rolling(window=window_size, min_periods=1).mean()

# Calculate the rolling average of anglez
merged_data['enmo_rolling_avg'] = merged_data['enmo'].rolling(window=window_size, min_periods=1).mean()

print(merged_data.head(10))

         series_id    step                 timestamp     anglez    enmo  \
5203  8e32047cbc1f   39600  2017-09-18T00:00:00-0400 -70.946701  0.0223   
1680  29c75c018220  198360  2018-04-16T00:00:00-0400 -27.437000  0.0105   
1694  29c75c018220  319320  2018-04-23T00:00:00-0400 -37.416100  0.0173   
2967  4feda0596965  231492  2018-05-26T00:01:00-0400  16.277399  0.0025   
7401  ce9164297046  230772  2018-02-07T00:01:00-0500  12.339900  0.0042   
2973  4feda0596965  317892  2018-05-31T00:01:00-0400   9.275100  0.0000   
8151  de6fedfb6139  441732  2018-05-05T00:01:00-0400 -30.678699  0.0040   
7187  c908a0ad3e31   22332  2018-02-04T00:01:00-0500 -43.887798  0.0306   
5101  8a306e0890c0  368112  2017-11-23T00:01:00-0500 -58.055199  0.0181   
9547  fe90110788d2  263892  2017-08-20T00:01:00-0400 -72.914101  0.0124   

      night  event timestamp_notimezone             timestamp_utc      time  \
5203      3  onset  2017-09-18T00:00:00 2017-09-18 00:00:00+00:00  00:00:00   
1680     12  ons

## 5. Model Selection

In [9]:
# Convert event to numerical labels (onset: 0, wakeup: 1) for classification
merged_data['event_label'] = merged_data['event'].map({'onset': 0, 'wakeup': 1})

print(merged_data.head(10))

         series_id    step                 timestamp     anglez    enmo  \
5203  8e32047cbc1f   39600  2017-09-18T00:00:00-0400 -70.946701  0.0223   
1680  29c75c018220  198360  2018-04-16T00:00:00-0400 -27.437000  0.0105   
1694  29c75c018220  319320  2018-04-23T00:00:00-0400 -37.416100  0.0173   
2967  4feda0596965  231492  2018-05-26T00:01:00-0400  16.277399  0.0025   
7401  ce9164297046  230772  2018-02-07T00:01:00-0500  12.339900  0.0042   
2973  4feda0596965  317892  2018-05-31T00:01:00-0400   9.275100  0.0000   
8151  de6fedfb6139  441732  2018-05-05T00:01:00-0400 -30.678699  0.0040   
7187  c908a0ad3e31   22332  2018-02-04T00:01:00-0500 -43.887798  0.0306   
5101  8a306e0890c0  368112  2017-11-23T00:01:00-0500 -58.055199  0.0181   
9547  fe90110788d2  263892  2017-08-20T00:01:00-0400 -72.914101  0.0124   

      night  event timestamp_notimezone             timestamp_utc      time  \
5203      3  onset  2017-09-18T00:00:00 2017-09-18 00:00:00+00:00  00:00:00   
1680     12  ons

In [10]:
# Model Selection
model = RandomForestClassifier(n_estimators=100, random_state=42)

## 6. Training 

In [11]:
# Training
X = merged_data[['hour', 'anglez_rolling_avg', 'enmo_rolling_avg']]
y = merged_data['event_label']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [12]:
from sklearn.metrics import accuracy_score

y_pred = model.predict(X_val)
accuracy = accuracy_score(y_val, y_pred)
print("Validation Accuracy:", accuracy)


Validation Accuracy: 0.9801773604590506


## 7. Classification on test data 

In [13]:
path3 = "/kaggle/input/child-mind-institute-detect-sleep-states/test_series.parquet"
test_series = pd.read_parquet(path3)
print(test_series.head(100))

       series_id  step                 timestamp     anglez    enmo
0   038441c925bb     0  2018-08-14T15:30:00-0400   2.636700  0.0217
1   038441c925bb     1  2018-08-14T15:30:05-0400   2.636800  0.0215
2   038441c925bb     2  2018-08-14T15:30:10-0400   2.637000  0.0216
3   038441c925bb     3  2018-08-14T15:30:15-0400   2.636800  0.0213
4   038441c925bb     4  2018-08-14T15:30:20-0400   2.636800  0.0215
..           ...   ...                       ...        ...     ...
95  038441c925bb    95  2018-08-14T15:37:55-0400 -80.013603  0.0128
96  038441c925bb    96  2018-08-14T15:38:00-0400 -80.007599  0.0136
97  038441c925bb    97  2018-08-14T15:38:05-0400 -80.136703  0.0135
98  038441c925bb    98  2018-08-14T15:38:10-0400 -80.111801  0.0132
99  038441c925bb    99  2018-08-14T15:38:15-0400 -80.051201  0.0130

[100 rows x 5 columns]


In [14]:
# Remove timezone information from the timestamp (e.g., "-0400")
test_series['timestamp_notimezone'] = test_series['timestamp'].str[:-5]

# Convert timestamp to datetime
test_series['timestamp_utc'] = pd.to_datetime(test_series['timestamp_notimezone'],utc=True)

# Extract time from timestamp
test_series['time'] = test_series['timestamp_utc'].dt.time

# Extract time from timestamp
test_series['hour'] = test_series['timestamp_utc'].dt.hour

# Extract day of the week from timestamp and convert to string (e.g., 'Monday', 'Tuesday', etc.)
# test_series['day_of_week'] = test_series['timestamp_utc'].dt.day_name()

test_series = test_series.drop(columns=['timestamp','timestamp_notimezone','timestamp_utc'])

print(test_series.head(100))

       series_id  step     anglez    enmo      time  hour
0   038441c925bb     0   2.636700  0.0217  15:30:00    15
1   038441c925bb     1   2.636800  0.0215  15:30:05    15
2   038441c925bb     2   2.637000  0.0216  15:30:10    15
3   038441c925bb     3   2.636800  0.0213  15:30:15    15
4   038441c925bb     4   2.636800  0.0215  15:30:20    15
..           ...   ...        ...     ...       ...   ...
95  038441c925bb    95 -80.013603  0.0128  15:37:55    15
96  038441c925bb    96 -80.007599  0.0136  15:38:00    15
97  038441c925bb    97 -80.136703  0.0135  15:38:05    15
98  038441c925bb    98 -80.111801  0.0132  15:38:10    15
99  038441c925bb    99 -80.051201  0.0130  15:38:15    15

[100 rows x 6 columns]


In [15]:
# Sort the DataFrame based on timestamp
test_series = test_series.sort_values(by='time')

print(test_series.head(100))

        series_id  step     anglez    enmo      time  hour
150  03d92c9f6f8a     0  38.892899  0.0803  12:00:00    12
151  03d92c9f6f8a     1  29.374399  0.0752  12:00:05    12
152  03d92c9f6f8a     2  37.225101  0.1791  12:00:10    12
153  03d92c9f6f8a     3  46.937000  0.0922  12:00:15    12
154  03d92c9f6f8a     4  60.486698  0.0342  12:00:20    12
..            ...   ...        ...     ...       ...   ...
245  03d92c9f6f8a    95 -88.216599  0.0000  12:07:55    12
246  03d92c9f6f8a    96 -88.216599  0.0000  12:08:00    12
247  03d92c9f6f8a    97 -88.216599  0.0000  12:08:05    12
248  03d92c9f6f8a    98 -88.216599  0.0000  12:08:10    12
249  03d92c9f6f8a    99 -88.216599  0.0000  12:08:15    12

[100 rows x 6 columns]


In [16]:
window_size = 10

# Calculate the rolling average of anglez
test_series['anglez_rolling_avg'] = test_series['anglez'].rolling(window=window_size, min_periods=1).mean()

test_series = test_series.drop(columns=['anglez'])

# Calculate the rolling average of anglez
test_series['enmo_rolling_avg'] = test_series['enmo'].rolling(window=window_size, min_periods=1).mean()

test_series = test_series.drop(columns=['enmo'])


print(test_series.head(10))

        series_id  step      time  hour  anglez_rolling_avg  enmo_rolling_avg
150  03d92c9f6f8a     0  12:00:00    12           38.892899          0.080300
151  03d92c9f6f8a     1  12:00:05    12           34.133649          0.077750
152  03d92c9f6f8a     2  12:00:10    12           35.164133          0.111533
153  03d92c9f6f8a     3  12:00:15    12           38.107350          0.106700
154  03d92c9f6f8a     4  12:00:20    12           42.583220          0.092200
155  03d92c9f6f8a     5  12:00:25    12           44.011983          0.089267
156  03d92c9f6f8a     6  12:00:30    12           41.416928          0.086814
157  03d92c9f6f8a     7  12:00:35    12           39.996275          0.082425
158  03d92c9f6f8a     8  12:00:40    12           38.433878          0.085144
159  03d92c9f6f8a     9  12:00:45    12           33.281560          0.082110


In [17]:
test_features = test_series[['hour', 'anglez_rolling_avg', 'enmo_rolling_avg']]

# Get predicted class probabilities
class_probabilities = model.predict_proba(test_features)


# Get the predicted labels (the class with the highest probability)
predicted_labels = model.classes_[class_probabilities.argmax(axis=1)]

# Get the confidence scores (the highest class probability for each sample)
confidence_scores = class_probabilities.max(axis=1)

# Append the predictions and confidence scores to the test series DataFrame
test_series['predicted_event_label'] = predicted_labels
test_series['event'] = test_series['predicted_event_label'].map({0: 'onset', 1: 'wakeup'})
test_series['score'] = confidence_scores

# Add a new column 'row_id' that increments by 1 for each row, starting from 1
test_series.insert(0, 'row_id', range(0, len(test_series)))

test_series = test_series.drop(columns=['predicted_event_label','hour','anglez_rolling_avg','enmo_rolling_avg','time'])

# Save the DataFrame with predictions and confidence scores to a CSV file
test_series.to_csv('submission.csv', index=False)
